# COMP47670 Assignment 1 - Autumn 2026/27
## Task 1: Data Collection

**Student Name**: Ellen Lyons

**Student Number**: 26117753

In this first task, I will use web-scraping techniques to collect and parse data from the data source '*Property Sale Register*'. 

First I had a look at the ``robots.txt`` file for the website http://mlg.ucd.ie/robots.txt and here is what it said:

```User-agent: *
disallow: /files/*

User-agent: *
disallow: /datasets/comp30120/*

User-agent: *
disallow: /datasets/comp41450/*

User-agent: *
disallow: /datasets/comp41680/*
```

Ok so I can't scrape any paths that start with ```/files/``` or ```/datasets/``` but the website I am scraping is  ```http://mlg.ucd.ie/modules/python/assign1/property/index.html```, so we are all good. 

(Here is where I got my explanation of robots.txt files: https://medium.com/@mikram2015/understanding-robots-txt-the-beginners-guide-for-web-scrapers-e4643d4e8f38)

In [27]:
import bs4 
import urllib.request

I am going to use the *Beautiful Soup* package to extract HTML info about the property sales. Throughout this assignment, I will often use statements like ``try``/``except`` or ``if``/``else`` so that if there is an error retrieivng data, or an error within a file, my selected warning will be printed rather than Python printing me an error message. 

This is just good practise, and again, I am following this from the lectures/example code. 

 ### Challenges & Solutions

1. The first thing I did was figure out how to parse a single page of property information. I wrote this as a function ``parse_one_page(url)``.  

The structure of the page is that there are ```<li>``` tags that tell us the date the property was sold, and then within this tag are ```<table> ``` tags that contains cells that tell us about the price, location, year built, etc. This caused a challenge because it is not straightforward to access the data I want. The solution was to make lots of ```for``` loops and print out lots of things to figure out where I was going wrong.


2. I messed around on the website I chose to see how it is formatted. It looks something like this:

```

> Homepage
    > 2022
        > pages 1 - 17
    > 2023
        > pages 1 - 16
    > 2024
        > pages 1 - 18
    > 2025
        > pages 1 - 23

```
So I needed to find a way to loop over each year, and loop over each page within each year. The number of pages is different for each year. This is why I started by writing a function that parses an individual page, because it made it so much easier to write the loops over the years and all pages. 

In [ ]:
def parse_one_page(url): # function to parse a single page of property sales
    
    # first get the HTML of the page
    try:
        response = urllib.request.urlopen(url)
        html = response.read().decode("utf-8")
        #print(html)
    except Exception as e:
        #print(f"Error retrieving page: {e}")
        
        return None

    property_info = [] # list to store results

    # use beautifulsoup to parse the HTML
    soup = bs4.BeautifulSoup(html, 'html.parser')
    #print(soup.prettify())

    sale_matches = soup.find_all("li") # find all li tags we want

    if not sale_matches: # check if there are any results
        print("No property sale records found")

    else:
        #print(f"Processing {len(sale_matches)} house sale records ...")
        
        for i, match in enumerate(sale_matches, 1):
            # within each property, find span tag and extract text from each one
            date_sold_elem = match.find("span", {"class": "sold"}) # there is only one span tag!
            
            # check all required elements are found
            if date_sold_elem is None:
                print(f"Warning: Property sale record {i} is missing required fields")
                continue
            
            # extract text 
            date_sold = date_sold_elem.text.strip()
            
            # add info to a dictionary
            property_sale = {"Date Sold": date_sold}
            
            # find the table with more property info
            table = match.find("table", {"class": "sale"})

            if table is None:
                print(f"No further info about property {i} found")
                continue
            #print(table.prettify())
            
            # extract info from each row in the table
            rows = table.find_all("tr") # tr stands for table row 

            for row in rows:
                cells = row.find_all("td") # td stands for table data cell

                # --- I got a bit stuck as to how to loop over the rows so I used the logic from this person:
                # ----- Source - https://stackoverflow.com/a/51657193
                # ----- Posted by Andrej Kesely
                # ----- Retrieved 2026-10-21, License - CC BY-SA 4.0

                if len(cells) == 2: # there should only be two 'td' tags
                    key = cells[0].text.strip()
                    value = cells[1].text.strip()
                    #print(key, value)
                    property_sale[key] = value

                else:
                    print(f"Issue with number of data cells in property {i} ...")           

            property_info.append(property_sale)

    return property_info


I will test if the function works by trying out different years and pages. 

In [22]:
test_url = "http://mlg.ucd.ie/modules/python/assign1/property/2022-page01.html"

parse_one_page(test_url)

[{'Date Sold': 'Sold 26 December 2022',
  'Sale Price:': '€359,086.00',
  'Location:': 'Oak Park',
  'Year Built:': '1984',
  'Size:': '1,131sqft',
  'Garden:': 'Yes',
  'Garage:': 'Yes',
  'Property Details:': 'Type: Detached; Style: 1.5-Storey; 3 Bedrooms; 1 Bathroom',
  'First Time Buyer:': 'Yes',
  'Property Reference:': 'PROP-105204'},
 {'Date Sold': 'Sold 2022-12-25',
  'Sale Price:': '€734,419.00',
  'Location:': 'Porter Square',
  'Year Built:': '2019',
  'Size:': '1,941sqft',
  'Garden:': 'Yes',
  'Garage:': 'Yes',
  'Property Details:': 'Type: Detached; 2 Bathrooms; Style: 2-Storey; 3 Bedrooms',
  'First Time Buyer:': 'NO',
  'Property Reference:': 'PROP-105203'},
 {'Date Sold': 'Sold 2022-12-19',
  'Sale Price:': '€724,835.00',
  'Location:': 'Porter Square',
  'Year Built:': '2021',
  'Size:': '1,996 sqft',
  'Garden:': 'Yes',
  'Garage:': 'Yes',
  'Property Details:': 'Type: Det.; Style: 2-Storey; 3 Bedrooms; 2 Bathrooms',
  'First Time Buyer:': 'No',
  'Property Reference

In [ ]:
# looping over years and pages to extract all property sale records!

BASE_URL = "http://mlg.ucd.ie/modules/python/assign1/property/{}-page{:02d}.html" # the :02 d menas to format the page number as 01, 02, 03, etc 

all_property_info = [] 

for year in range(2022, 2026): # loop over year 2022 to 2025
    
    page_num = 1
    
    while True:
        results = parse_one_page(BASE_URL.format(year,page_num)) 

        if not results:
            break
        
        all_property_info.extend(results)
        page_num += 1

print(f"Successfully extracted {len(all_property_info)} property sale records!")



Successfully extracted 1450 property sale records!
